# Gemma 4 Particle Edu
### Free 3D Physics Simulation Platform powered by Gemma 4 + Ollama

**Tracks**: Future of Education / Ollama Special Technology / Main

| | |
|---|---|
| Live Demo | [gemma4-particle-edu.vercel.app](https://gemma4-particle-edu.vercel.app) |
| Video | [YouTube (3 min)](https://youtu.be/3e-LZPHBA2M) |
| GitHub | [U2SY26/gemma4-particle-edu](https://github.com/U2SY26/gemma4-particle-edu) |
| Benchmark Data | [Kaggle Dataset](https://www.kaggle.com/datasets/syu21125/gemma4-particle-edu-benchmark-300) |

## The Problem

Interactive physics simulations are locked behind expensive tools. Claude Artifacts charges per conversation. PhET simulations are pre-built with no customization. Students can't type "show me DNA" and watch it form with real physics.

We built a platform where you describe any physics scenario in natural language, and an AI generates a real-time 3D particle simulation with accurate SI-unit physics. Completely free. Runs locally on your laptop with Ollama.

## Architecture

```
User Input (natural language)
       |
       v
DAG Pipeline (5 steps)
  1. ANALYZE  -- identify object, domain, scale
  2. RESEARCH -- look up exact SI physical values
  3. DESIGN   -- plan particle layout and connections
  4. GENERATE -- produce simulation JSON
  5. VALIDATE -- self-check physics accuracy
       |
       v
Verlet Integration Physics Engine
  - 25,000 particles
  - SI units (gravity, density, temperature)
  - Spring connections for structural cohesion
       |
       v
Three.js WebGL Renderer
  - Neon bloom post-processing
  - Real-time 60fps
```

The DAG pipeline is the key innovation. Instead of one LLM call that often hallucinates physics values, we chain 5 specialized steps where each step's output feeds the next. This raised accuracy from 84% (single call) to 99.4% (DAG chain).

## How Gemma 4 is Used

Gemma 4 31B (Q4_K_M, 20.3GB) runs locally via Ollama. It handles all 5 DAG steps:

- **Step 1 (ANALYZE)**: Classifies the user's request into physics domain, identifies key properties
- **Step 2 (RESEARCH)**: Retrieves exact physical constants with a verified material physics table (45+ materials) injected into the prompt
- **Step 3 (DESIGN)**: Plans the particle arrangement using shapes (helix, sphere, grid, torus, etc.) and connections
- **Step 4 (GENERATE)**: Produces the final simulation JSON with all physics parameters
- **Step 5 (VALIDATE)**: Self-checks the generated JSON against physical reality and corrects errors

For web deployment, the same pipeline runs with Gemini 2.5 Pro as a fallback when Ollama is unavailable.

## Benchmark: 300 Scenarios, 138 Materials, 99.4% Accuracy

We ran 300 diverse physics scenarios through the full pipeline and validated each with Verlet integration (100 frames).

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Find benchmark data (handles different Kaggle dataset slug formats)
input_dir = '/kaggle/input'
data_path = None
if os.path.exists(input_dir):
    for d in os.listdir(input_dir):
        candidate = os.path.join(input_dir, d, 'benchmark-300.json')
        if os.path.exists(candidate):
            data_path = candidate
            break

if not data_path:
    print('Dataset not found in /kaggle/input, using embedded summary data')
    data = {'summary': {'total': 300, 'perfect': 293, 'avgAccuracy': 99.4, 'materials': 138, 'exploded': 6, 'model': 'gemma4:31b', 'runtime': '17h 43m', 'gpu': 'RTX 5090'}, 'scenarios': []}
else:
    with open(data_path) as f:
        data = json.load(f)

print(f"Summary: {json.dumps(data['summary'], indent=2)}")

if data['scenarios']:
    df = pd.DataFrame(data['scenarios'])
    print(f"\nTotal scenarios: {len(df)}")
    print(f"Perfect (100%): {(df['accuracy']==100).sum()}")
    print(f"Unique materials: {df['material'].nunique()}")
    df.head(10)

In [ ]:
if data['scenarios']:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. Accuracy breakdown
    acc_counts = df['accuracy'].value_counts().sort_index()
    colors = ['#3fb950' if a == 100 else '#f0883e' if a >= 80 else '#f85149' for a in acc_counts.index]
    axes[0].bar(acc_counts.index.astype(str), acc_counts.values, color=colors)
    axes[0].set_title('Accuracy Distribution')
    axes[0].set_xlabel('Accuracy (%)')
    axes[0].set_ylabel('Count')
    for i, (x, v) in enumerate(zip(acc_counts.index, acc_counts.values)):
        axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

    # 2. Star rating
    star_counts = df['stars'].value_counts().sort_index()
    axes[1].bar(star_counts.index, star_counts.values, color=['#f85149','#f0883e','#3fb950'])
    axes[1].set_title('Star Rating Distribution')
    axes[1].set_xlabel('Stars')

    # 3. Stable vs Exploded
    exploded = df['exploded'].value_counts()
    axes[2].pie(exploded.values, labels=['Stable','Exploded'], autopct='%1.1f%%',
                colors=['#3fb950','#f85149'], startangle=90)
    axes[2].set_title('Simulation Stability')

    plt.tight_layout()
    plt.show()
else:
    print('293/300 perfect (97.7%), 6 exploded (all extreme astrophysics), avg accuracy 99.4%')

In [ ]:
if data['scenarios']:
    # Top 20 materials
    mat = df.groupby('material').agg(count=('id','count'), avg_acc=('accuracy','mean'), boom=('exploded','sum')).sort_values('count', ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['#f85149' if a < 100 else '#3fb950' for a in mat['avg_acc']]
    ax.barh(mat.index[::-1], mat['count'][::-1], color=colors[::-1])
    ax.set_xlabel('Number of Scenarios')
    ax.set_title('Top 20 Materials (red = imperfect accuracy)')
    for i, (cnt, acc) in enumerate(zip(mat['count'][::-1], mat['avg_acc'][::-1])):
        ax.text(cnt + 0.3, i, f'{acc:.0f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
if data['scenarios']:
    failures = df[df['accuracy'] < 100]
    print(f'All {len(failures)} failures (all extreme astrophysics):\n')
    for _, r in failures.iterrows():
        print(f"  #{r['id']} {r['title'][:50]}  material={r['material']}  gravity={r['gravity']}  acc={r['accuracy']}%  exploded={r['exploded']}")

All 7 failures share one pattern: extreme astrophysical conditions with gravity at 10^12 m/s2 or higher (black holes, neutron stars, supernovae). The Verlet integration engine diverges at these scales. The parameters Gemma 4 generated are physically correct -- it's the simulation engine's numerical limit.

For all normal physics (engineering, biology, chemistry, fluids, room-temperature): **100% accuracy across 293 scenarios.**

## Model Comparison: 8B vs 31B

| Metric | Gemma 4 8B | Gemma 4 31B |
|--------|-----------|------------|
| Parameters | 8B (Q4_K_M) | 31B (Q4_K_M) |
| File size | 5.6 GB | 20.3 GB |
| VRAM | ~6 GB | ~22 GB |
| Pipeline | Single call | 5-step DAG |
| JSON success | 100% | 100% |
| Physics accuracy | 84.4% | **99.4%** |

The jump from 84% to 99% comes from: the larger model's better physics knowledge, and the DAG pipeline that breaks the problem into focused steps.

## Educational Impact

**Future of Education track:**
1. 99.4% physics accuracy -- students won't learn wrong values
2. Transparent 5-step reasoning visible to students, modeling scientific thinking
3. Free and local via Ollama -- no API costs, works offline, data stays private
4. 138 materials from elementary science to graduate research
5. Experiential -- students watch physics happen, not read about it

**Ollama track:**
Running Gemma 4 locally via Ollama produces benchmark-validated results at scale. 300 scenarios, 17 hours, zero API cost. No internet required.

## Tech Stack

| Layer | Technology |
|-------|------------|
| AI | Gemma 4 31B via Ollama / Gemini 2.5 Pro (web) |
| Physics | Verlet integration, SI units |
| Rendering | Three.js, WebGL, neon bloom |
| Server | Express.js + Vercel Functions |
| Frontend | Vanilla JS, i18n (Korean + English) |

~13,600 lines of code, 20 files. No frameworks beyond Three.js and Express.

## Links

- Live Demo: https://gemma4-particle-edu.vercel.app
- Video (3 min): https://youtu.be/3e-LZPHBA2M
- GitHub: https://github.com/U2SY26/gemma4-particle-edu
- Benchmark Dataset: https://www.kaggle.com/datasets/syu21125/gemma4-particle-edu-benchmark-300
- 31-page Report PDF: included in the dataset above